In [ ]:
# Standard imports
import tensorflow as tf
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Masking, Input, Dropout, Concatenate
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, brier_score_loss, classification_report, confusion_matrix
from sklearn.calibration import calibration_curve

## Import ICU By-Day Dataframe

In [ ]:
df_by_day = pd.read_pickle('micu_by_day.pkl')

# Ensure in_time is in datetime format
df_by_day['in_time'] = pd.to_datetime(df_by_day['in_time'])

# Check shape of dataframe
print("Sequential Data Shape:", df_by_day.shape)

## Data Transformation and Preprocessing

### Define Column Groups

In [ ]:
target_col = 'extended_stay'
cols_to_drop = ['stay_id', 'icu_day', 'los"', 'extended_stay', 'in_time']
cat_cols = ['gender', 'race', 'admission_type', 'admission_location']
num_cols = ['age', 'daily_heart_rate', 'daily_resp_rate', 'daily_spo2', 'daily_sys_bp', 'daily_temp']
feature_cols = cat_cols + num_cols

## Split Data into Train, Validation, and Test Sets

### Establish dates for splitting chronologically

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code that executes training, validation, and test splits 
#### in chronological order, with the oldest data being part of the training set 
#### and the most recent data being part of the test set."
#### Usage: Used quantile approach for splitting data using in_time.
#### -------------------------------------------------------------------------

In [ ]:
# Get the earliest in time for each patient
patient_timeline = df_by_day.groupby('stay_id')['in_time'].min().sort_values().reset_index()

# Identify the cutoff dates (using 70% of the data for training and 15% for validation)
train_cutoff = patient_timeline['in_time'].quantile(0.70)
val_cutoff = patient_timeline['in_time'].quantile(0.85)

# Ensure formatting is correct
train_end_date = train_cutoff.strftime('%Y-%m-%d')
val_end_date = val_cutoff.strftime('%Y-%m-%d')

# Slice data using identfied cutoff dates
df_train = df_by_day[df_by_day['in_time'] <= train_end_date]. copy()
df_val = df_by_day[(df_by_day['in_time'] > train_end_date) & (df_by_day['in_time'] <= val_end_date)].copy()
df_test = df_by_day[df_by_day['in_time'] > val_end_date].copy()

# Reset indices for clean splits
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

### Build Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ],
    remainder="drop"
)

### Fit Preprocessor & Transform Data

In [ ]:
# Isolate features
X_train = df_train[feature_cols]
X_val = df_val[feature_cols]
X_test = df_test[feature_cols]

# Train preprocessor
preprocessor.fit(X_train)

# Fit preprocessor
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Convert data to floats
X_train_processed = np.asarray(X_train_processed, dtype=np.float32)
X_val_processed = np.asarray(X_val_processed, dtype=np.float32)
X_test_processed = np.asarray(X_test_processed, dtype=np.float32)

# Save preprocessor for streamlit application
joblib.dump(preprocessor, "preprocessor.pkl", protocol=4)

### Reshape Data to 3D for LSTM Input

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code to properly reshape data from 2D to 3D to 
#### prepare for LSTM model training. The data should be grouped by patient
#### and sorted in sequential day order."
#### Usage: Used reshaping function approach with recommended data shape
#### standardization.
#### -------------------------------------------------------------------------

In [ ]:
def reshape_to_3d(df, X_matrix, max_days=4):
    # Initiate empty sequences list
    sequences = []
    # Group by patient
    grouped = df.groupby("stay_id")
    for s, patient_df in grouped:
        # Sort by time
        patient_df = patient_df.sort_values("icu_day")
        # Extract preprocessed features
        idx = patient_df.index.to_numpy()
        patient_X = np.asarray(X_matrix[idx], dtype=np.float32)
        # Standardize shape 
        if patient_X.shape[0] < max_days:
            pad = np.zeros((max_days - patient_X.shape[0], patient_X.shape[1]), dtype=np.float32)
            patient_X = np.vstack([patient_X, pad])
        else:
            patient_X = patient_X[:max_days]
        sequences.append(patient_X)
    # Return 3D data
    return np.array(sequences, dtype=np.float32)

### Execute 3D Inputs

In [ ]:
X_train_3d = reshape_to_3d(df_train, X_train_processed)
X_val_3d = reshape_to_3d(df_val, X_val_processed)
X_test_3d = reshape_to_3d(df_test, X_test_processed)

### Execute Vitals vs. Static Data Split

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code that takes training, validation, and test data 
#### that has been reshaped to 3D and split it into two different sets of data. 
#### The vitals (dynamic) variables should be one split, and the static variables 
#### should be another split."
#### Usage: Used numeric/static splitting approach on training set; repeated on
#### validation and test sets.
#### -------------------------------------------------------------------------

In [ ]:
# Define split point
num_size = len(num_cols)

# Keep all patients, all days, and only the dynamic columns
X_train_num = X_train_3d[:, :, :num_size]
# Keep all patients, only day 1, and only the static columns
X_train_static = X_train_3d[:, 0, num_size:]

# Execute slice for validation set
X_val_num = X_val_3d[:, :, :num_size]
X_val_static = X_val_3d[:, 0, num_size:]

# Execute slice for test set
X_test_num = X_test_3d[:, :, :num_size]
X_test_static = X_test_3d[:, 0, num_size:]

### Generate Target Labels

In [ ]:
y_train = df_train.groupby("stay_id")[target_col].first().values
y_val = df_val.groupby("stay_id")[target_col].first().values
y_test = df_test.groupby("stay_id")[target_col].first().values

### Create Model

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code that creates an LSTM model that can take in both 
#### evolving vitals data and static patient data. Masking needs to be applied, 
#### and I'd like to include dropout to prevent overfitting."
#### Usage: Used overall Model approach (initially tried Sequential() but ran
#### into many issues/errors); used recommended dropout values initially,
#### but tweaked them as I saw fit when training the model.
#### -------------------------------------------------------------------------

In [ ]:
# Takes 3D shape as input
input_num = Input(shape=(4, len(num_cols)))
# Takes 2D shape as input
input_static = Input(shape=(X_train_static.shape[1],))

# Tell model to ignore 0 values
vitals_processed = Masking(mask_value=0.0)(input_num)
# Process vitals
vitals_processed = LSTM(64)(vitals_processed)
# Prevent overfitting by randomly ignoring 40% of neurons during training
vitals_processed = Dropout(0.4)(vitals_processed)

# Create static path
static_processed = Dense(16, activation="relu")(input_static)

# Combine vital and static data
merged_features = Concatenate()([vitals_processed, static_processed])
dense_refined = Dense(32, activation="relu")(merged_features)
# Prevent overfitting by randomly ignoring 20% of neurons during training
dense_refined = Dropout(0.2)(dense_refined)

# Set up for binary classification
output = Dense(1, activation="sigmoid")(dense_refined)

lstm = Model([input_num, input_static], output)

### Compile Model

In [ ]:
# Reduced learning rate implemented for more stable convergence / improved generalization
lstm.compile(
    optimizer=tf.keras.optimizers.Adam(0.0005),
    loss="binary_crossentropy",
    metrics=["AUC"]
)

### Train Model

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

hist = lstm.fit(
    [X_train_num, X_train_static],
    y_train,
    validation_data=([X_val_num, X_val_static], y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop]
)

## Evaluate Performance on Validation Set

### Plot Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(hist.history['loss'], label="Training Loss")
plt.plot(hist.history['val_loss'], label="Validation Loss")
plt.title("LSTM Model Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Generate Validation Predictions

In [ ]:
y_val_prob = lstm.predict([X_val_num, X_val_static]).ravel()
y_val_pred = (y_val_prob >= 0.5).astype(int)

### Validation ROC-AUC and AUPRC

In [ ]:
val_auc = roc_auc_score(y_val, y_val_prob)
val_auprc = average_precision_score(y_val, y_val_prob)

print(f"Validation ROC-AUC: {val_auc:.4f}")
print(f"Validation AUPRC: {val_auprc:.4f}")

#### Given that extended stays make up 25% of the data, **relative improvement from this model's AUPRC score is 236%**.

### Validation ROC Curve

In [ ]:
# Calculate validation false positive rate, true positive tate, and thresholds
fpr_val, tpr_val, thresholds_val = roc_curve(y_val, y_val_prob)

# Plot the validation ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr_val, tpr_val, label=f"Validation ROC (AUC = {val_auc:.3f})")

# Plot the random guess baseline
plt.plot([0, 1], [0, 1], label='Random Guess (AUC = 0.500)')

# Format plot
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation Set Receiver Operating Characteristic (ROC)")
plt.legend(loc='lower right')
plt.show()

### Validation Brier Score

In [ ]:
val_brier = brier_score_loss(y_val, y_val_prob)
print(f"Brier Score Loss: {val_brier:.4f}")

#### Given that extended stays make up 25% of the data, **relative improvement from this model's Brier score is 69%**.

### Validation Calibration Curve (Reliability Diagram)

In [ ]:
# Compute calibration curve data
prob_true_lstm_val, prob_pred_lstm_val = calibration_curve(y_val, y_val_prob, n_bins=10, strategy='quantile')

# Plot curve
plt.figure(figsize=(7, 7))
plt.plot([0,1], [0,1], "k--", label="Perfect Calibration")
plt.plot(prob_pred_lstm_val, prob_true_lstm_val, 's-', label=f"LSTM (Brier: {val_brier:.4f})")

# Format plot
plt.xlabel("Predicted Probability of Extended Stay")
plt.ylabel("Actual Extended Stay Rate")
plt.title("LSTM Calibration Curve (Validation Set)")
plt.legend(loc='lower right')
plt.show()

### Validation Youden's J Score

In [ ]:
val_j = tpr_val - fpr_val

# Find the index of the highest J score
best_idx_val = np.argmax(val_j)
best_threshold_val = thresholds_val[best_idx_val]
best_j_value_val = val_j[best_idx_val]

print(f"Validation Optimal J Score Threshold: {best_threshold_val:.4f}")
print(f"Validation Maximized Youden's J Score: {best_j_value_val:.4f}")

## Evaluate Performance on Test Set

### Generate Test Predictions

In [ ]:
y_test_prob = lstm.predict([X_test_num, X_test_static]).ravel()
y_test_pred = (y_test_prob >= best_threshold_val).astype(int)

### Test Metrics

In [ ]:
# Calculate test false positive rate, true positive tate, and thresholds
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_test_prob)

# Calculate final test metrics
test_roc = roc_auc_score(y_test, y_test_prob)
test_auprc = average_precision_score(y_test, y_test_prob)
test_brier = brier_score_loss(y_test, y_test_prob)

print(f"Final Test ROC-AUC: {test_roc:.4f}")
print(f"Final Test AUPRC: {test_auprc:.4f}")
print(f"Final Test Brier Loss: {test_brier:.4f}")

print("Final Test Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=["Standard Stay", "Extended Stay"]))

print("Final Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

#### **Final relative AUPRC improvement over baseline is 240%**.

#### **Final relative Brier score improvement over baseline is 71%**.

### Test Calibration Curve (Reliability Diagram)

In [ ]:
# Compute calibration curve data
prob_true_lstm_test, prob_pred_lstm_test = calibration_curve(y_test, y_test_prob, n_bins=10, strategy='quantile')

# Plot curve
plt.figure(figsize=(7, 7))
plt.plot([0,1], [0,1], "k--", label="Perfect Calibration")
plt.plot(prob_pred_lstm_test, prob_true_lstm_test, 's-', label=f"LSTM (Brier: {test_brier:.4f})")

# Format plot
plt.xlabel("Predicted Probability of Extended Stay")
plt.ylabel("Actual Extended Stay Rate")
plt.title("LSTM Calibration Curve (Test Set)")
plt.legend(loc='lower right')
plt.show()

## Save Final Model to Export

In [ ]:
lstm.save('lstm_model_new.keras')